Golden Hospital Discharge Record
------------------------------------------------------------

In [1]:
import pandas as pd

In [2]:
Clinical_Findings = pd.read_excel(r"C:\Users\Veman S Chippa\OneDrive\Desktop\Dhire Assesment\Question_5_dataset\clinical_findings.xlsx")

In [3]:
Clinical_Findings.head()

,IPID,DIS_DATE,PID,DEPARTMENT,GENDER,AGE,CLINICALDESCRIPTION,CLINICALDATA,ADVICEDMEDICINE
0,585,2021-02-01,310002,ONCOLOGY,F,66 Ys 4 Ms,DSC_PRESENTHISTORY,"Case of Ca ovary ,post 1st line chemotherapy",NaN
1,585,2021-02-01,310002,ONCOLOGY,F,66 Ys 4 Ms,DSC_FINALDIAGNOSISS,CA OVARY POST 1ST LINE CT,NaN
2,597,2021-02-01,236525,ONCOLOGY,F,62 Ys 0 Ms,DSC_CLINICALFINDINGS,O/E : BP : 100/70mmHg Pulse : 96/mt CVS : S1S2...,1.\tTAB TAXIM-O 200MG 1-0-1 X 5 DAYS\n2.\tTAB....
3,597,2021-02-01,236525,ONCOLOGY,F,62 Ys 0 Ms,DSC_PRESENTHISTORY,Mrs.Geetha is a known case of Advanced CA Ovar...,1.\tTAB TAXIM-O 200MG 1-0-1 X 5 DAYS\n2.\tTAB....
4,597,2021-02-01,236525,ONCOLOGY,F,62 Ys 0 Ms,DSC_FINALDIAGNOSISS,CA OVARY – STAGE IV ?ACUTE COLITIS LEFT LOWER ...,1.\tTAB TAXIM-O 200MG 1-0-1 X 5 DAYS\n2.\tTAB....


In [4]:
Discharge_Details = pd.read_excel(r"C:\Users\Veman S Chippa\OneDrive\Desktop\Dhire Assesment\Question_5_dataset\discharge_details.xlsx")

In [5]:
Discharge_Details.head()

,IPID,DIS_DATE,PID,PROCEDURECATEGORY,AMOUNT
0,4817,2021-06-27,262839,ASCITES TAAPPING,1000.0
1,7821,2021-09-27,357830,ASSISTING CHARGE,11500.0
2,10858,2021-12-02,224106,ANESTHESIA CHARGE,2000.0
3,12324,2021-12-25,364829,ASWINI LAB,1500.0
4,8751,2021-10-18,359691,ASWINI LAB,600.0


In [6]:
Discharge_Master = pd.read_excel(r"C:\Users\Veman S Chippa\OneDrive\Desktop\Dhire Assesment\Question_5_dataset\discharge_master.xlsx")

In [7]:
Discharge_Master.head()

,IPID,PID,ADMITTED,DISCHARGED,DEPARTMENT,GROSSAMT,DISCOUNT,NETAMT,INSURANCE,PATIENTPAYABLE
0,1295,301714,2021-02-23,2021-02-23,ONCOLOGY,2740.48,0.00,2740.48,0.0,2740
1,1283,319580,2021-02-23,2021-02-23,ONCOLOGY,35872.50,13052.18,22820.32,0.0,22820
2,1304,250260,2021-02-23,2021-02-23,ONCOLOGY,7101.30,1252.00,5849.30,0.0,5849
3,1151,310916,2021-02-18,2021-02-24,ONCOLOGY,36936.77,0.00,36936.77,0.0,21937
4,1336,321249,2021-02-24,2021-02-24,ONCOLOGY,10078.44,5336.72,4741.72,0.0,4742


In [8]:
Discharge_Master['ADMITTED'] = pd.to_datetime(Discharge_Master['ADMITTED'])
Discharge_Master['DISCHARGED'] = pd.to_datetime(Discharge_Master['DISCHARGED'])

Discharge_Master['LENGTH_OF_STAY'] = (
    Discharge_Master['DISCHARGED'] - Discharge_Master['ADMITTED']
).dt.days

In [9]:
Discharge_Master.head()

,IPID,PID,ADMITTED,DISCHARGED,DEPARTMENT,GROSSAMT,DISCOUNT,NETAMT,INSURANCE,PATIENTPAYABLE,LENGTH_OF_STAY
0,1295,301714,2021-02-23,2021-02-23,ONCOLOGY,2740.48,0.00,2740.48,0.0,2740,0
1,1283,319580,2021-02-23,2021-02-23,ONCOLOGY,35872.50,13052.18,22820.32,0.0,22820,0
2,1304,250260,2021-02-23,2021-02-23,ONCOLOGY,7101.30,1252.00,5849.30,0.0,5849,0
3,1151,310916,2021-02-18,2021-02-24,ONCOLOGY,36936.77,0.00,36936.77,0.0,21937,6
4,1336,321249,2021-02-24,2021-02-24,ONCOLOGY,10078.44,5336.72,4741.72,0.0,4742,0


In [10]:
Discharge_Details['AMOUNT'] = Discharge_Details['AMOUNT'].astype(float)

In [11]:
details_agg = (
    Discharge_Details
    .groupby('IPID')
    .agg(
        TOTAL_PROCEDURE_AMOUNT=('AMOUNT', 'sum'),
        PROCEDURE_COUNT=('AMOUNT', 'count')
    )
    .reset_index()
)


In [12]:
details_agg

,IPID,TOTAL_PROCEDURE_AMOUNT,PROCEDURE_COUNT
0,585,17524.00,11
1,597,8058.00,6
2,625,4690.00,4
3,626,20872.00,6
4,630,32149.00,13
...,...,...,...
4274,91660,3386.93,3
4275,91710,14640.50,4
4276,91716,12861.47,3
4277,91720,22749.03,6


In [13]:
clinical_basic = (
    Clinical_Findings
    .groupby('IPID')
    .agg(
        GENDER=('GENDER', 'first'),
        AGE=('AGE', 'first')
    )
    .reset_index()
)

In [14]:
final_diagnosis = (
    Clinical_Findings[
        Clinical_Findings['CLINICALDESCRIPTION'] == 'DSC_FINALDIAGNOSISS'
    ]
    .sort_values('DIS_DATE')
    .groupby('IPID')
    .first()
    .reset_index()[['IPID', 'CLINICALDATA']]
    .rename(columns={'CLINICALDATA': 'FINAL_DIAGNOSIS'})
)


In [15]:
clinical_summary = (
    Clinical_Findings
    .groupby('IPID')['CLINICALDATA']
    .apply(lambda x: ' | '.join(x.dropna()))
    .reset_index(name='CLINICAL_SUMMARY')
)


In [16]:
golden_record = (
    Discharge_Master
    .merge(details_agg, on='IPID', how='left')
    .merge(clinical_basic, on='IPID', how='left')
    .merge(final_diagnosis, on='IPID', how='left')
    .merge(clinical_summary, on='IPID', how='left')
)


In [17]:
golden_record

,IPID,PID,ADMITTED,DISCHARGED,DEPARTMENT,GROSSAMT,DISCOUNT,NETAMT,INSURANCE,PATIENTPAYABLE,LENGTH_OF_STAY,TOTAL_PROCEDURE_AMOUNT,PROCEDURE_COUNT,GENDER,AGE,FINAL_DIAGNOSIS,CLINICAL_SUMMARY
0,1295,301714,2021-02-23,2021-02-23,ONCOLOGY,2740.48,0.00,2740.48,0.00,2740,0,2740.00,3.0,NaN,NaN,NaN,NaN
1,1283,319580,2021-02-23,2021-02-23,ONCOLOGY,35872.50,13052.18,22820.32,0.00,22820,0,35872.00,4.0,NaN,NaN,NaN,NaN
2,1304,250260,2021-02-23,2021-02-23,ONCOLOGY,7101.30,1252.00,5849.30,0.00,5849,0,7101.00,4.0,NaN,NaN,NaN,NaN
3,1151,310916,2021-02-18,2021-02-24,ONCOLOGY,36936.77,0.00,36936.77,0.00,21937,6,36937.00,16.0,F,61 Ys 7 Ms,DLBCL STAGE IV STATUS POST CHEMO+RT - REFRACTO...,GC : MODERATE PALLOR +VE PS:II SPO2: 80% OFF O...
4,1336,321249,2021-02-24,2021-02-24,ONCOLOGY,10078.44,5336.72,4741.72,0.00,4742,0,10078.00,4.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4275,91660,463985,2025-06-29,2025-06-29,ONCOLOGY,3386.93,0.00,3386.93,0.00,3387,0,3386.93,3.0,NaN,NaN,NaN,NaN
4276,91720,485545,2025-06-30,2025-06-30,ONCOLOGY,22749.03,7257.25,15491.78,13177.00,2315,0,22749.03,6.0,NaN,NaN,NaN,NaN
4277,91716,496010,2025-06-30,2025-06-30,ONCOLOGY,12861.47,4839.15,8022.32,8022.32,0,0,12861.47,3.0,NaN,NaN,NaN,NaN
4278,91710,515229,2025-06-30,2025-06-30,ONCOLOGY,14640.50,4177.70,10462.80,0.00,8463,0,14640.50,4.0,NaN,NaN,NaN,NaN


In [18]:
assert golden_record.shape[0] == Discharge_Master.shape[0], \
"Row count mismatch: Golden record must have one row per IPID"

assert golden_record['IPID'].is_unique, \
"Duplicate IPIDs found in golden record"

In [19]:
golden_record.to_csv('golden_discharge_record.csv', index=False)

In [20]:
golden_record.shape[0] == Discharge_Master.shape[0]

True